# Laya vs Jev, judged on the same task, on Strands
Same 500 items, same two questions, zero-shot. Every number below is computed live in this run.

The Laya and Jev sections are the original notebook, unchanged. What moves to [Strands Agents](https://strandsagents.com)
is Claude: the reference judge now runs on Amazon Bedrock as a Strands agent with a typed reply (`reference_judge.py`),
and the last section runs the cascade live, with Jev in front of Claude as a Strands hook.

In [ ]:
from nb_helpers import *
versions()

## The data, built on first run
Nothing is shipped. Haiku writes 500 technical questions and answers, one in three clean and the rest with one planted
defect, and the reference judge (Claude Sonnet 5) grades each one blind. Both run on Bedrock through Strands
(`build_dataset.py`). The first run takes about 10 minutes and costs about $5; after that this cell only confirms the
files are complete, because both stages skip the ids they already hold.

In [ ]:
from build_dataset import main as build_data
build_data(500)

## The 500 items and the reference judge
Claude Sonnet graded every item blind, earlier. **Agreement** below means agreement with it, not correctness.

In [ ]:
items = load_items()
corpus_summary(items)

## Two questions, identical for both candidates

In [ ]:
print(VERDICT_OPTIONS)
print(len(CRITERIA), 'failure modes, e.g.', list(CRITERIA)[1:6])

## Laya, local on Apple silicon
`laya-mlx` is a community runtime, not written by Convai.

In [ ]:
laya = load_laya()
time_laya(laya, items)

In [ ]:
laya_verdict = run_laya(laya, items, {'verdict': QUESTIONS['verdict']})
score(laya_verdict, items, 'verdict')

## Was I holding it wrong? Four framings, first 200 items

In [ ]:
four_framings(laya, items[:200])

## The 40-option question, and my mistake

In [ ]:
budget_check(laya)

In [ ]:
laya_criterion = criterion_three_ways(laya, items)

## Jev, hosted API

In [ ]:
await jev_single_call(items)

In [ ]:
jev = await run_jev(items)
score(jev, items, 'verdict'); score(jev, items, 'criterion')

In [ ]:
jev_short = await run_jev_short(items)
score(jev_short, items, 'criterion')

## What Laya's confidence field actually is (laya 0.3.5)

In [ ]:
show_confidence_source()

## The reference judge, live, as a Strands agent on Bedrock
The 500 labels were built at the top of this notebook. Here the same judge grades the first 20 items again: Claude Sonnet 5 on
Bedrock, one fresh Strands agent per item, replying with the typed `Grade` model instead of JSON in text.
Claude is not deterministic, so a regrade will not match its own recorded labels on every item.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from reference_judge import GOLD_MODEL, Grade, ask, gold_prompt


def judge(item, hooks=None):
    """One item through the reference judge. `state` is Strands' invocation_state, shared with any hooks."""
    state = {"item": item}
    grade, usage = ask(GOLD_MODEL, gold_prompt(item), Grade, hooks=hooks, state=state)
    return {"id": item["id"], "grade": grade, "jev": state.get("jev"), **usage}


sample = items[:20]
with ThreadPoolExecutor(4) as pool:
    live = list(pool.map(judge, sample))

same_verdict = sum(r["grade"].verdict == it["gold_verdict"] for r, it in zip(live, sample))
same_criterion = sum(r["grade"].criterion == it["gold_criterion"] for r, it in zip(live, sample))
claude_usd_per_item = sum(r["in"] * 3 + r["out"] * 15 for r in live) / len(live) / 1e6
print(f"{GOLD_MODEL}, {len(sample)} items regraded")
print(f"matches its recorded label: verdict {same_verdict}/{len(sample)}, criterion {same_criterion}/{len(sample)}")
print(f"{sum(r['in'] for r in live) / len(live):.0f} in / {sum(r['out'] for r in live) / len(live):.0f} out per item "
      f"= ${claude_usd_per_item * 1000:.2f} per 1,000 (the typed reply adds its schema to every call)")

## How much of the grading bill can I delete?
Accept the most confident answers unreviewed, send the rest to the reference judge.

In [ ]:
cascade_table({
    'Jev': (jev, jev['usd'] / len(items)),
    'Laya, 2 options': (laya_verdict, 0.0),
    'Laya, 40 options, budget 1024': (laya_criterion['long descriptions, budget 1024'], 0.0),
}, items)

## The cascade, live: Jev in front of Claude, as a Strands hook
The table above is hindsight maths over recorded runs. Here it runs for real. `JevFirst` is a hook on the reference
judge's agent: on `BeforeInvocationEvent` it asks Jev, and if Jev's verdict is confident enough it keeps that verdict
and cancels the invocation, so Claude is never called. Otherwise the call goes through and Claude grades as usual.
The judge, its prompt and its typed reply are not touched.

The confidence cut is the one the cascade maths found on the full Jev run above: the lowest confidence at which the
most confident items still agree with the reference judge 95% of the time. Verdict only, the high-volume question.

In [ ]:
import compare
from strands.hooks import BeforeInvocationEvent, HookProvider, HookRegistry
from typesafe_sdk import TypeSafeClient

from judge_task import jev_questions, state_for

ranked = sorted(jev["rows"], key=lambda r: r["verdict_conf"], reverse=True)
cut = compare.cascade(jev["rows"], {r["id"]: r for r in items}, "verdict", jev["usd"] / len(items))
ACCEPT_AT = ranked[round(cut["coverage"] * len(ranked)) - 1]["verdict_conf"] if cut["coverage"] else 1.01
print(f"accept Jev's verdict at confidence >= {ACCEPT_AT:.3f}   (covers {cut['coverage']:.1%} of the full run)")


class JevFirst(HookProvider):
    """Before the reference judge runs, ask Jev. A confident Jev verdict is kept and the Claude call is cancelled."""

    def __init__(self, accept_at: float) -> None:
        self.accept_at, self.jev, self.questions = accept_at, TypeSafeClient(), jev_questions()

    def register_hooks(self, registry: HookRegistry, **_) -> None:
        registry.add_callback(BeforeInvocationEvent, self.ask_jev)

    def ask_jev(self, event: BeforeInvocationEvent) -> None:
        item = event.invocation_state["item"]
        response = self.jev.system_one(state=state_for(item["question"], item["answer"]), questions=self.questions)
        verdict = response.answers["verdict"]
        event.invocation_state["jev"] = {
            "verdict": verdict.choice, "confidence": float(verdict.confidence), "tokens": response.usage.input_tokens or 0,
        }
        if verdict.confidence >= self.accept_at:
            event.cancel = f"Jev: {verdict.choice}"

In [ ]:
import time

hook = JevFirst(ACCEPT_AT)
sample = items[:100]
start = time.perf_counter()
with ThreadPoolExecutor(8) as pool:
    runs = list(pool.map(lambda item: judge(item, hooks=[hook]), sample))
elapsed = time.perf_counter() - start

by_jev = [(r, it) for r, it in zip(runs, sample) if r["grade"] is None]
by_claude = [(r, it) for r, it in zip(runs, sample) if r["grade"] is not None]
agree = lambda pairs, pick: sum(pick(r) == it["gold_verdict"] for r, it in pairs)
jev_hits = agree(by_jev, lambda r: r["jev"]["verdict"])
claude_hits = agree(by_claude, lambda r: r["grade"].verdict)
usd = sum((r["in"] * 3 + r["out"] * 15) / 1e6 + r["jev"]["tokens"] * 0.042 / 1e6 for r in runs) / len(runs)

print(f"{len(sample)} items in {elapsed:.0f} s   Claude called for {len(by_claude)}, Jev answered {len(by_jev)}")
print(f"agreement with the recorded reference: {(jev_hits + claude_hits) / len(sample):.1%} overall   "
      f"Jev-accepted {jev_hits}/{len(by_jev)}   Claude-graded {claude_hits}/{len(by_claude)}")
print(f"${usd * 1000:.2f} per 1,000 with the cascade vs ${claude_usd_per_item * 1000:.2f} all Claude   "
      f"bill deleted {1 - usd / claude_usd_per_item:.1%}")

Read the live cascade with the same care as the table: 100 items is a small sample, the cut was chosen on the full
run (in-sample), and the Claude-graded branch is scored against Claude's own earlier labels, so its misses are
Claude disagreeing with itself.